In [0]:
%run ../source_to_bronze/utils

In [0]:
from pyspark.sql.types import *

In [0]:
employee_df = spark.table("sample.default.employee_q_1")
department_df = spark.table("sample.default.department_q_1")
country_df = spark.table("sample.default.country_q_1")

employee_df.printSchema()
department_df.printSchema()
country_df.printSchema()

In [0]:
display(employee_df)
display(department_df)
display(country_df)

In [0]:
employee_schema = StructType([
    StructField("EmployeeID", LongType(), True),
    StructField("EmployeeName", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Salary", LongType(), True),
    StructField("Age", LongType(), True)
])

In [0]:
department_schema = StructType([
    StructField("DepartmentID", StringType(), True),
    StructField("DepartmentName", StringType(), True)
])

In [0]:
country_schema = StructType([
    StructField("CountryCode", StringType(), True),
    StructField("CountryName", StringType(), True)
])

In [0]:
employee_bronze_df = (
    spark.read
    .option("header", True)
    .schema(employee_schema)
    .csv("/Volumes/sample/default/assignment_volume/source_to_bronze/employee_df")
)

In [0]:
display(employee_bronze_df)
employee_bronze_df.printSchema()

In [0]:
department_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(department_schema)
    .load("/Volumes/sample/default/assignment_volume/source_to_bronze/department_df")
)

In [0]:
display(department_bronze_df)
department_bronze_df.printSchema()

In [0]:
country_bronze_df = read_csv(
    "/Volumes/sample/default/assignment_volume/source_to_bronze/country_df",
    country_schema
)

In [0]:
display(country_bronze_df)
country_bronze_df.printSchema()

In [0]:
employee_silver_df = convert_columns_to_snake_case(employee_bronze_df)

department_silver_df = convert_columns_to_snake_case(department_bronze_df)

country_silver_df = convert_columns_to_snake_case(country_bronze_df)

In [0]:
employee_silver_df.printSchema()
department_silver_df.printSchema()
country_silver_df.printSchema()

In [0]:
employee_silver_df = add_load_date(employee_silver_df)

department_silver_df = add_load_date(department_silver_df)

country_silver_df = add_load_date(country_silver_df)

In [0]:
display(employee_silver_df)
display(department_silver_df)
display(country_silver_df)

In [0]:
silver_path = "/Volumes/sample/default/assignment_volume/silver/Employee_info/dim_employee"

In [0]:
(
    employee_silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

In [0]:
silver_employee_df = spark.read.format("delta").load(silver_path)

display(silver_employee_df)

In [0]:
silver_employee_df.printSchema()

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/sample/default/assignment_volume/silver/Employee_info/dim_employee`